# FEATURE SELECTION - VN INDEX

## Import libraries

In [25]:
import os
import sys
import random
from datetime import datetime, timedelta
import pandas as pd
import lightning as L
import torch.nn as nn
import torch
from torch.optim import Adam
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tsfresh.utilities.dataframe_functions import (
    roll_time_series,
)
from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import ComprehensiveFCParameters, EfficientFCParameters
import matplotlib.pylab as plt
from sklearn.preprocessing import MinMaxScaler
from lightning.pytorch.callbacks import EarlyStopping
from dataclasses import asdict
import ipynbname
from sklearn.inspection import permutation_importance
import xgboost as xgb
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.base import clone

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import *
from dtos.config_dtos.config_dto import ConfigDto
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from utils.enums import (
    LossFunctionType,
    ModelAchitectureType,
    OptimizerType,
    ScalerType,
)
from ta.ta_functions import *

In [26]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Helper functions

In [27]:
def get_weekends(from_date: str, to_date: str):
    start = datetime.strptime(from_date, "%Y-%m-%d")
    end = datetime.strptime(to_date, "%Y-%m-%d")

    weekends = []
    current = start

    while current <= end:
        if current.weekday() in (5, 6):  # 5 = Saturday, 6 = Sunday
            weekends.append(current.strftime("%Y-%m-%d"))
        current += timedelta(days=1)

    return weekends

## Parameters

In [28]:
STOCK_NAME = "vn_index"
NOTEBOOK_NAME = ipynbname.name()
RANDOM_SEED = 18
MAX_TIMESHIFT = 5
MIN_TIMESHIFT = 5
FORECAST_HORIZON = 5
COLUMN_ID = "stock"
TARGET_COLUMN = f"close_{FORECAST_HORIZON}"
DATE_COLUMN = "date"
FEATURE_COLUMNS = []  # all TA indicators

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2021-12-31")
VALIDATION_RANGE = ("2022-01-01", "2023-12-31")
TEST_RANGE = ("2024-01-01", "2026-02-26")

In [29]:
WEEKENDS = get_weekends(TRAIN_RANGE[0], TEST_RANGE[1])
HOLIDAYS = []

DAYOFFS = []
DAYOFFS.extend(WEEKENDS)
DAYOFFS.extend(HOLIDAYS)

DAYOFFS[:10], DAYOFFS[-10:]

(['2000-01-01',
  '2000-01-02',
  '2000-01-08',
  '2000-01-09',
  '2000-01-15',
  '2000-01-16',
  '2000-01-22',
  '2000-01-23',
  '2000-01-29',
  '2000-01-30'],
 ['2026-01-24',
  '2026-01-25',
  '2026-01-31',
  '2026-02-01',
  '2026-02-07',
  '2026-02-08',
  '2026-02-14',
  '2026-02-15',
  '2026-02-21',
  '2026-02-22'])

## Load data

In [30]:
my_logger = Logger(file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/vn_index/test")

In [31]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [32]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [33]:
vn_index_df = my_postgresql_driver.select(
    schema_name="stock_market", table_name="vn_index"
)

In [34]:
dtype_map = {
    "date": str,
    "open": float,
    "high": float,
    "low": float,
    "close": float,
    "adjust": float,
    "change": float,
    "percent_change": float,
    "matching_volume": float,
    "matching_value": float,
    "negotiate_volume": float,
    "negotiate_value": float,
    "number_of_buy_orders": float,
    "buy_volume": float,
    "average_volume_per_buy_order": float,
    "number_of_sell_orders": float,
    "sell_volume": float,
    "average_volume_per_sell_order": float,
    "net_volume": float,
}

vn_index_df = (
    vn_index_df.astype(dtype_map)
    .dropna(subset=["close"])
    .sort_values(by=["date"])
    .reset_index(drop=True)
)

In [35]:
vn_index_df

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970000,4.740000,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.000000,3448.0,9.057040e+06,2627.0,4.362437e+07
1,2008-03-08,646.190,646.190,646.190,646.190,646.190,25.363333,4.106667,1.314382e+07,8.298321e+11,769057.0,4.594466e+10,27442.0,5.068024e+07,1852.333333,7486.0,1.712815e+07,2464.0,3.355209e+07
2,2008-03-09,652.240,652.240,652.240,652.240,652.240,21.756667,3.473333,1.811842e+07,1.144410e+12,1188073.0,6.278627e+10,28068.0,4.867907e+07,1739.666667,11523.0,2.519927e+07,2301.0,2.347980e+07
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150000,2.840000,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.000000,15561.0,3.327038e+07,2138.0,1.340752e+07
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580000,-2.970000,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.000000,17910.0,2.811427e+07,1570.0,-8.782960e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6561,2026-02-22,1832.317,1859.236,1829.801,1856.535,1856.535,33.445000,1.837000,7.158068e+08,2.227241e+13,23581058.0,7.924402e+11,476711.0,1.233848e+09,2591.300000,393692.0,1.198445e+09,3047.9,3.540296e+07
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050000,1.980000,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.000000,401281.0,1.213944e+09,3025.0,3.906029e+07
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480000,0.400000,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.000000,507471.0,1.648273e+09,3248.0,-7.304628e+07
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710000,-0.360000,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.000000,597096.0,1.892734e+09,3170.0,-3.036983e+07


## Data transformation

In [36]:
vn_index_df.shape

(6566, 19)

### Remove DAYOFFS

In [37]:
vn_index_df_t1 = vn_index_df[~vn_index_df["date"].isin(DAYOFFS)]
vn_index_df_t1

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.0,3448.0,9.057040e+06,2627.0,4.362437e+07
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.0,15561.0,3.327038e+07,2138.0,1.340752e+07
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.0,17910.0,2.811427e+07,1570.0,-8.782960e+06
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,290070.0,2.036917e+10,16836.0,2.378011e+07,1412.0,12032.0,2.034621e+07,1691.0,3.433900e+06
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,289160.0,1.702626e+10,13298.0,1.703506e+07,1281.0,12256.0,1.900562e+07,1551.0,-1.970560e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,24358775.0,8.224590e+11,456926.0,1.195538e+09,2623.9,378513.0,1.167449e+09,3093.7,2.808830e+07
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.0,401281.0,1.213944e+09,3025.0,3.906029e+07
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.0,507471.0,1.648273e+09,3248.0,-7.304628e+07
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.0,597096.0,1.892734e+09,3170.0,-3.036983e+07


### Create target

In [38]:
vn_index_df_t2 = vn_index_df_t1.copy()
vn_index_df_t2[f"close_{FORECAST_HORIZON}"] = vn_index_df_t1["close"].shift(
    -FORECAST_HORIZON
)
vn_index_df_t2

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume,close_5
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.0,3448.0,9.057040e+06,2627.0,4.362437e+07,643.80
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.0,15561.0,3.327038e+07,2138.0,1.340752e+07,615.71
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.0,17910.0,2.811427e+07,1570.0,-8.782960e+06,588.26
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,290070.0,2.036917e+10,16836.0,2.378011e+07,1412.0,12032.0,2.034621e+07,1691.0,3.433900e+06,573.45
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,289160.0,1.702626e+10,13298.0,1.703506e+07,1281.0,12256.0,1.900562e+07,1551.0,-1.970560e+06,564.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,24358775.0,8.224590e+11,456926.0,1.195538e+09,2623.9,378513.0,1.167449e+09,3093.7,2.808830e+07,NaN
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.0,401281.0,1.213944e+09,3025.0,3.906029e+07,NaN
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.0,507471.0,1.648273e+09,3248.0,-7.304628e+07,NaN
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.0,597096.0,1.892734e+09,3170.0,-3.036983e+07,NaN


### Create features

In [39]:
DF_MAP = {}
DF_MAP

{}

#### add_bbands

In [40]:
DF_MAP[add_bbands.__name__] = {}
vn_index_df_add_bbands = add_bbands(vn_index_df_t2, n=list(range(2, 21)))
DF_MAP[add_bbands.__name__]["dataframe"] = vn_index_df_add_bbands
vn_index_df_add_bbands

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_bb_20_bandwidth_slope,close_bb_20_bandwidth_acceleration,close_bb_20_pct_b,close_bb_20_pct_b_slope,close_bb_20_pct_b_gt_1,close_bb_20_pct_b_lt_0,close_bb_20_above_upper,close_bb_20_below_lower,close_bb_20_inside_bands,close_bb_20_position
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-0.003677,0.002547,0.818102,0.053949,False,False,False,False,True,0
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.002366,0.006043,0.887543,0.069440,False,False,False,False,True,0
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.004535,0.002169,0.905573,0.018031,False,False,False,False,True,0
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.003006,-0.001528,0.819302,-0.086272,False,False,False,False,True,0


#### add_dema

In [41]:
DF_MAP[add_dema.__name__] = {}
vn_index_df_add_dema = add_dema(vn_index_df_t2, n=list(range(2, 21)))
DF_MAP[add_dema.__name__]["dataframe"] = vn_index_df_add_dema
vn_index_df_add_dema

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_dema_18_19_direction,close_dema_18_19_dist_slope,close_dema_18_20_dist,close_dema_18_20_dist_abs,close_dema_18_20_direction,close_dema_18_20_dist_slope,close_dema_19_20_dist,close_dema_19_20_dist_abs,close_dema_19_20_direction,close_dema_19_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-1,0.279618,-0.325177,0.325177,-1,0.550689,-0.266359,0.266359,-1,0.271071
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1,0.298477,0.263810,0.263810,1,0.588987,0.024151,0.024151,1,0.290510
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,0.271562,0.803720,0.803720,1,0.539909,0.292498,0.292498,1,0.268347
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,0.109531,1.035922,1.035922,1,0.232202,0.415169,0.415169,1,0.122671


In [42]:
list(DF_MAP.keys())

['add_bbands', 'add_dema']

In [43]:
DF_MAP[add_bbands.__name__]

{'dataframe':             date      open      high       low     close    adjust  change  \
 0     2008-03-07   640.140   640.140   640.140   640.140   640.140  28.970   
 3     2008-03-10   658.290   658.290   658.290   658.290   658.290  18.150   
 4     2008-03-11   638.710   638.710   638.710   638.710   638.710 -19.580   
 5     2008-03-12   643.900   643.900   643.900   643.900   643.900   5.190   
 6     2008-03-13   647.600   647.600   647.600   647.600   647.600   3.700   
 ...          ...       ...       ...       ...       ...       ...     ...   
 6559  2026-02-20  1829.151  1851.448  1821.603  1849.325  1849.325  28.235   
 6562  2026-02-23  1833.900  1863.130  1833.900  1860.140  1860.140  36.050   
 6563  2026-02-24  1862.180  1867.690  1849.600  1867.620  1867.620   7.480   
 6564  2026-02-25  1869.490  1876.010  1855.890  1860.910  1860.910  -6.710   
 6565  2026-02-26  1861.160  1882.170  1859.040  1879.640  1879.640  18.730   
 
       percent_change  matching_volum

## Filter columns

In [44]:
print(
    f"Create feature columns: {[DATE_COLUMN, TARGET_COLUMN, "close", "open", "high", "low", "adjust"]}"
)

for df_name in DF_MAP.keys():
    print(f"\nProcessing dataframe {df_name}")

    DF_MAP[df_name]["feature_columns"] = [
        col
        for col in DF_MAP[df_name]["dataframe"].columns
        if col
        not in [DATE_COLUMN, TARGET_COLUMN, "close", "open", "high", "low", "adjust"]
    ]

Create feature columns: ['date', 'close_5', 'close', 'open', 'high', 'low', 'adjust']

Processing dataframe add_bbands

Processing dataframe add_dema


In [45]:
DF_MAP[add_bbands.__name__]["dataframe"]

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_bb_20_bandwidth_slope,close_bb_20_bandwidth_acceleration,close_bb_20_pct_b,close_bb_20_pct_b_slope,close_bb_20_pct_b_gt_1,close_bb_20_pct_b_lt_0,close_bb_20_above_upper,close_bb_20_below_lower,close_bb_20_inside_bands,close_bb_20_position
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-0.003677,0.002547,0.818102,0.053949,False,False,False,False,True,0
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.002366,0.006043,0.887543,0.069440,False,False,False,False,True,0
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.004535,0.002169,0.905573,0.018031,False,False,False,False,True,0
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.003006,-0.001528,0.819302,-0.086272,False,False,False,False,True,0


## Prepare data

In [46]:
print(f"Create X and y dataframes")

for df_name, cfg in DF_MAP.items():
    print(f"\nProcessing dataframe {df_name}")

    df = cfg["dataframe"]

    # Ensure datetime + sort
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
    df = df.sort_values(by=DATE_COLUMN).reset_index(drop=True)

    cfg["dataframe"] = df

    # Train / Validation split
    train_df = df[df[DATE_COLUMN].between(TRAIN_RANGE[0], TRAIN_RANGE[1])]
    val_df = df[df[DATE_COLUMN].between(VALIDATION_RANGE[0], VALIDATION_RANGE[1])]

    cfg["train_dataframe"] = train_df
    cfg["val_dataframe"] = val_df

    # Features / target
    feature_cols = cfg["feature_columns"]

    cfg["X_train"] = train_df[feature_cols]
    cfg["y_train"] = train_df[TARGET_COLUMN]

    cfg["X_val"] = val_df[feature_cols]
    cfg["y_val"] = val_df[TARGET_COLUMN]

Create X and y dataframes

Processing dataframe add_bbands

Processing dataframe add_dema


## Model

In [47]:
# Model hyperparameters
MODEL_N_ESTIMATORS = 200  # 5000
MODEL_MAX_DEPTH = 20
MODEL_LEARNING_RATE = 0.01
MODEL_SUBSAMPLE = 0.6
MODEL_COLSAMPLE_BYTREE = 0.5
MODEL_MIN_CHILD_WEIGHT = 30
MODEL_REG_ALPHA = 1.0
MODEL_REG_LAMBDA = 10.0

# Training / system parameters
MODEL_TREE_METHOD = "hist"
MODEL_DEVICE = "cuda"
MODEL_EARLY_STOPPING_ROUNDS = 100
MODEL_ENABLE_CATEGORICAL = True
MODEL_RANDOM_STATE = RANDOM_SEED

N_REPEATS = 1  # 100
TOP_N = 100
CORR_THRESHOLD = 0.95

In [48]:
model = xgb.XGBRegressor(
    n_estimators=MODEL_N_ESTIMATORS,
    max_depth=MODEL_MAX_DEPTH,
    learning_rate=MODEL_LEARNING_RATE,
    subsample=MODEL_SUBSAMPLE,
    colsample_bytree=MODEL_COLSAMPLE_BYTREE,
    min_child_weight=MODEL_MIN_CHILD_WEIGHT,
    reg_alpha=MODEL_REG_ALPHA,
    reg_lambda=MODEL_REG_LAMBDA,
    tree_method=MODEL_TREE_METHOD,
    device=MODEL_DEVICE,
    early_stopping_rounds=MODEL_EARLY_STOPPING_ROUNDS,
    enable_categorical=MODEL_ENABLE_CATEGORICAL,
    random_state=MODEL_RANDOM_STATE,
)

model

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.5
,device,'cuda'
,early_stopping_rounds,100
,enable_categorical,True
,eval_metric,None


## Feature Importance + Feature Selection + Save Data

In [49]:
FEATURE_SELECTION_RESULT_DIR

'../../src/feature_selection/feature_selection_result'

In [50]:
os.makedirs(FEATURE_SELECTION_RESULT_DIR, exist_ok=True)

In [51]:
for df_name in DF_MAP.keys():
    print(f"\nProcessing dataframe: {df_name}")

    out_dir = os.path.join(FEATURE_SELECTION_RESULT_DIR, df_name)
    os.makedirs(out_dir, exist_ok=True)

    cfg = DF_MAP[df_name]
    df = cfg["dataframe"]

    model_copy = clone(model)
    model_copy.fit(
        cfg["X_train"],
        cfg["y_train"],
        eval_set=[(cfg["X_val"], cfg["y_val"])],
        verbose=100,
    )

    X = df[cfg["feature_columns"]]
    y = df[TARGET_COLUMN]

    # -----------------------------
    # BASELINE (GPU prediction)
    # -----------------------------
    print("BASELINE (GPU prediction)")
    baseline_pred = model_copy.predict(X)
    baseline_mse = np.mean((y - baseline_pred) ** 2)

    # -----------------------------
    # GPU PERMUTATION IMPORTANCE + PROGRESS
    # -----------------------------
    print("GPU PERMUTATION IMPORTANCE + PROGRESS")
    X_vals = X.values  # avoid repeated DataFrame overhead
    y_vals = y.values
    importances = []

    for i, col in enumerate(tqdm(X.columns, desc="Permutation Importance")):
        scores = np.empty(N_REPEATS)
        col_backup = X_vals[:, i].copy()

        for r in range(N_REPEATS):
            X_vals[:, i] = np.random.permutation(col_backup)
            pred = model_copy.predict(X_vals)
            scores[r] = np.mean((y_vals - pred) ** 2)

        X_vals[:, i] = col_backup  # restore column in-place
        importances.append(scores.mean() - baseline_mse)

    # -----------------------------
    # CREATE IMPORTANCE DATAFRAME
    # -----------------------------
    print("CREATE IMPORTANCE DATAFRAME")
    importance_df = (
        pd.DataFrame({"feature": X.columns, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    importance_df["importance_pct"] = (
        importance_df["importance"] / importance_df["importance"].sum() * 100
    )
    importance_df.to_csv(
        os.path.join(out_dir, f"{df_name}_importance_before.csv"), index=False
    )

    # -----------------------------
    # HELPER: plot top-N bar chart
    # -----------------------------
    def plot_top_n(imp_df, title, filename):
        top = (
            imp_df.sort_values("importance", ascending=False)
            .head(TOP_N)
            .set_index("feature")["importance"]
            .sort_values(ascending=True)
        )
        ax = top.plot(kind="barh", figsize=(12, 12))
        plt.title(title)
        plt.ylabel("Importance")
        ax.xaxis.set_ticks_position("both")
        ax.tick_params(axis="x", which="both", top=True, bottom=True, labeltop=True)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, filename), dpi=400, bbox_inches="tight")
        plt.close()

    # -----------------------------
    # HELPER: plot correlation heatmap
    # -----------------------------
    def plot_corr(features, title, filename):
        corr = df[features].corr()
        display(corr)
        plt.figure(figsize=(14, 12))
        sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=0.5)
        plt.title(title)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, filename), dpi=400, bbox_inches="tight")
        plt.close()

    # -----------------------------
    # TOP N FEATURE IMPORTANCE BEFORE REMOVAL
    # -----------------------------
    print("TOP N FEATURE IMPORTANCE BEFORE REMOVAL")
    plot_top_n(
        importance_df,
        f"Top {TOP_N} Feature Importance (Before)",
        "feature_importance_plot_before.png",
    )

    # -----------------------------
    # FEATURE CORRELATION BEFORE REMOVAL
    # -----------------------------
    print("FEATURE CORRELATION BEFORE REMOVAL")
    top_features_before = importance_df.head(TOP_N)["feature"].tolist()
    plot_corr(
        top_features_before,
        "Feature Correlation Matrix (Before)",
        "feature_correlation_plot_before.png",
    )

    # -----------------------------
    # REMOVE HIGHLY CORRELATED FEATURES
    # -----------------------------
    print("REMOVE HIGHLY CORRELATED FEATURES")
    corr_matrix_abs = df[cfg["feature_columns"]].corr().abs()
    corr_matrix_abs.to_csv(os.path.join(out_dir, f"{df_name}_corr_matrix_abs.csv"))
    sorted_features = importance_df[
        "feature"
    ].tolist()  # already sorted by importance desc

    selected_features, removed_features = [], set()
    for feature in sorted_features:
        if feature in removed_features:
            continue
        selected_features.append(feature)
        correlated = corr_matrix_abs.index[
            corr_matrix_abs[feature] > CORR_THRESHOLD
        ].tolist()
        removed_features.update(f for f in correlated if f != feature)

    print(f"Selected features: {len(selected_features)}")
    print(f"Removed features:  {len(removed_features)}")
    print(f"\nTop kept features:\n{selected_features[:TOP_N]}")
    print(f"\nRemoved features:\n{list(removed_features)}")

    # -----------------------------
    # TOP N FEATURE IMPORTANCE AFTER REMOVAL
    # -----------------------------
    print("TOP N FEATURE IMPORTANCE AFTER REMOVAL")
    importance_df_after = importance_df[
        importance_df["feature"].isin(selected_features)
    ].reset_index(drop=True)
    importance_df_after.to_csv(
        os.path.join(out_dir, f"{df_name}_importance_after.csv"), index=False
    )
    plot_top_n(
        importance_df_after,
        f"Top {TOP_N} Feature Importance (After)",
        "feature_importance_plot_after.png",
    )

    # -----------------------------
    # FEATURE CORRELATION AFTER REMOVAL
    # -----------------------------
    print("FEATURE CORRELATION AFTER REMOVAL")
    top_features_after = importance_df_after.head(TOP_N)["feature"].tolist()
    plot_corr(
        top_features_after,
        "Feature Correlation Matrix (After)",
        "feature_correlation_plot_after.png",
    )


Processing dataframe: add_bbands
[0]	validation_0-rmse:520.71858
[100]	validation_0-rmse:213.99112
[199]	validation_0-rmse:97.87398
BASELINE (GPU prediction)
GPU PERMUTATION IMPORTANCE + PROGRESS


d:\GIT\master-thesis\mt_env\Lib\site-packages\xgboost\core.py:158: UserWarning: [11:43:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
Permutation Importance: 100%|██████████| 450/450 [00:49<00:00,  9.15it/s]


CREATE IMPORTANCE DATAFRAME
TOP N FEATURE IMPORTANCE BEFORE REMOVAL
FEATURE CORRELATION BEFORE REMOVAL


,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,...,close_bb_5_slope_middle,close_bb_5_slope_middle_acceleration,close_bb_5_slope_lower,close_bb_5_slope_lower_acceleration,close_bb_5_bandwidth,close_bb_5_bandwidth_slope,close_bb_5_bandwidth_acceleration,close_bb_5_pct_b,close_bb_5_pct_b_slope,close_bb_5_pct_b_gt_1
change,1.000000,0.893572,-0.016419,-0.015606,0.024262,0.025782,-0.021873,0.048606,0.144190,0.021843,...,0.495547,0.670469,0.474282,0.393443,-0.099030,-0.199607,-0.154282,0.626521,0.570057,NaN
percent_change,0.893572,1.000000,-0.018403,-0.016684,0.008665,0.009581,-0.017961,0.032051,0.158959,0.004895,...,0.463246,0.606202,0.413931,0.333333,-0.081248,-0.178129,-0.138729,0.673097,0.587522,NaN
matching_volume,-0.016419,-0.018403,1.000000,0.963308,0.627757,0.534907,0.923316,0.860472,-0.046468,0.920885,...,0.097688,-0.058457,0.011950,-0.040996,-0.078981,0.042722,0.015694,0.069018,-0.054820,NaN
matching_value,-0.015606,-0.016684,0.963308,1.000000,0.570184,0.518406,0.938237,0.835739,-0.108362,0.928459,...,0.096674,-0.061023,0.013846,-0.043369,-0.068912,0.038808,0.016406,0.063401,-0.052319,NaN
negotiate_volume,0.024262,0.008665,0.627757,0.570184,1.000000,0.811036,0.586024,0.548093,-0.045510,0.598209,...,0.025602,0.004406,0.002763,-0.000351,-0.110138,0.010171,-0.001047,0.036149,0.016613,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
close_bb_5_bandwidth_slope,-0.199607,-0.178129,0.042722,0.038808,0.010171,0.022436,0.020402,0.017081,0.003573,0.020688,...,-0.120498,-0.141926,-0.728829,-0.522756,0.248885,1.000000,0.596967,-0.085538,-0.040374,NaN
close_bb_5_bandwidth_acceleration,-0.154282,-0.138729,0.015694,0.016406,-0.001047,0.009508,0.008131,0.001920,-0.006455,0.008764,...,-0.001008,-0.120752,-0.403810,-0.840665,-0.119759,0.596967,1.000000,-0.016893,-0.053720,NaN
close_bb_5_pct_b,0.626521,0.673097,0.069018,0.063401,0.036149,0.033029,0.030662,0.090586,0.203940,0.088889,...,0.624034,0.417024,0.433555,0.156338,-0.108787,-0.085538,-0.016893,1.000000,0.426151,NaN
close_bb_5_pct_b_slope,0.570057,0.587522,-0.054820,-0.052319,0.016613,0.018242,-0.047213,-0.020197,0.070445,-0.030009,...,-0.029339,0.437404,0.022018,0.205388,0.029690,-0.040374,-0.053720,0.426151,1.000000,NaN


REMOVE HIGHLY CORRELATED FEATURES
Selected features: 277
Removed features:  173

Top kept features:
['change', 'percent_change', 'matching_volume', 'negotiate_volume', 'negotiate_value', 'number_of_buy_orders', 'buy_volume', 'average_volume_per_buy_order', 'average_volume_per_sell_order', 'net_volume', 'close_bb_2_upper', 'close_bb_2_dist_upper', 'close_bb_2_dist_lower', 'close_bb_2_slope_upper', 'close_bb_2_slope_upper_acceleration', 'close_bb_2_slope_middle', 'close_bb_2_slope_middle_acceleration', 'close_bb_2_slope_lower', 'close_bb_2_slope_lower_acceleration', 'close_bb_2_bandwidth', 'close_bb_2_bandwidth_slope', 'close_bb_2_bandwidth_acceleration', 'close_bb_2_pct_b', 'close_bb_2_pct_b_slope', 'close_bb_2_pct_b_gt_1', 'close_bb_2_pct_b_lt_0', 'close_bb_2_above_upper', 'close_bb_2_below_lower', 'close_bb_2_inside_bands', 'close_bb_2_position', 'close_bb_3_dist_upper', 'close_bb_3_dist_middle', 'close_bb_3_dist_lower', 'close_bb_3_slope_upper', 'close_bb_3_slope_upper_acceleration',

...

Permutation Importance: 100%|██████████| 811/811 [02:20<00:00,  5.76it/s]


CREATE IMPORTANCE DATAFRAME
TOP N FEATURE IMPORTANCE BEFORE REMOVAL
FEATURE CORRELATION BEFORE REMOVAL


,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,...,close_dema_14_dist_abs,close_dema_15,close_dema_15_slope,close_dema_15_acceleration,close_gt_dema_15,close_dema_15_dist,close_dema_15_dist_abs,close_dema_16,close_dema_16_slope,close_dema_16_acceleration
change,1.000000,0.893572,-0.016419,-0.015606,0.024262,0.025782,-0.021873,0.048606,0.144190,0.021843,...,-0.234745,0.026619,0.612823,0.892672,0.413256,0.655164,-0.232871,0.026085,0.597992,0.896763
percent_change,0.893572,1.000000,-0.018403,-0.016684,0.008665,0.009581,-0.017961,0.032051,0.158959,0.004895,...,-0.177458,0.003193,0.566257,0.796971,0.441701,0.601492,-0.176749,0.002653,0.553137,0.800862
matching_volume,-0.016419,-0.018403,1.000000,0.963308,0.627757,0.534907,0.923316,0.860472,-0.046468,0.920885,...,0.324116,0.837980,0.102851,-0.080700,0.013632,-0.034470,0.322193,0.837761,0.109093,-0.080991
matching_value,-0.015606,-0.016684,0.963308,1.000000,0.570184,0.518406,0.938237,0.835739,-0.108362,0.928459,...,0.332422,0.854097,0.101584,-0.082721,0.009250,-0.046045,0.330745,0.853943,0.107873,-0.083095
negotiate_volume,0.024262,0.008665,0.627757,0.570184,1.000000,0.811036,0.586024,0.548093,-0.045510,0.598209,...,0.200650,0.660915,0.030321,0.013319,0.004408,-0.012548,0.201442,0.660769,0.032000,0.012868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
close_dema_15_dist,0.655164,0.601492,-0.034470,-0.046045,-0.012548,-0.017626,-0.049217,0.019682,0.121192,0.015133,...,-0.216762,-0.037713,0.879155,0.412041,0.680530,1.000000,-0.220239,-0.039187,0.863384,0.425607
close_dema_15_dist_abs,-0.232871,-0.176749,0.322193,0.330745,0.201442,0.205118,0.322109,0.270908,-0.138032,0.299186,...,0.996683,0.308312,-0.266935,-0.159400,-0.071758,-0.220239,1.000000,0.308660,-0.267083,-0.161387
close_dema_16,0.026085,0.002653,0.837761,0.853943,0.660769,0.602769,0.849847,0.740649,-0.092793,0.849656,...,0.308742,0.999996,0.067355,-0.016626,-0.002924,-0.039187,0.308660,1.000000,0.070821,-0.017158
close_dema_16_slope,0.597992,0.553137,0.109093,0.107873,0.032000,0.021135,0.063507,0.142262,0.173419,0.122631,...,-0.265906,0.072864,0.999312,0.260301,0.602957,0.863384,-0.267083,0.070821,1.000000,0.271180


REMOVE HIGHLY CORRELATED FEATURES
Selected features: 113
Removed features:  698

Top kept features:
['change', 'percent_change', 'matching_volume', 'negotiate_volume', 'negotiate_value', 'number_of_buy_orders', 'buy_volume', 'average_volume_per_buy_order', 'average_volume_per_sell_order', 'net_volume', 'close_dema_2', 'close_dema_2_acceleration', 'close_gt_dema_2', 'close_dema_2_dist', 'close_dema_2_dist_abs', 'close_dema_3_slope', 'close_gt_dema_3', 'close_dema_3_dist', 'close_dema_3_dist_abs', 'close_gt_dema_4', 'close_gt_dema_5', 'close_dema_5_dist', 'close_dema_5_dist_abs', 'close_dema_6_slope', 'close_gt_dema_6', 'close_gt_dema_7', 'close_dema_7_dist_abs', 'close_gt_dema_8', 'close_dema_8_dist', 'close_gt_dema_9', 'close_gt_dema_10', 'close_dema_10_dist_abs', 'close_dema_11_slope', 'close_gt_dema_11', 'close_gt_dema_12', 'close_dema_12_dist', 'close_gt_dema_14', 'close_dema_14_dist_abs', 'close_gt_dema_16', 'close_gt_dema_18', 'close_dema_18_dist', 'close_dema_19_dist_abs', 'close

,change,percent_change,matching_volume,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,average_volume_per_sell_order,net_volume,...,close_dema_5_20_dist_abs,close_dema_6_8_direction,close_dema_6_11_direction,close_dema_6_13_direction,close_dema_6_19_direction,close_dema_7_13_direction,close_dema_7_15_direction,close_dema_7_19_direction,close_dema_8_15_direction,close_dema_8_20_dist
change,1.000000,0.893572,-0.016419,0.024262,0.025782,-0.021873,0.048606,0.144190,-0.097816,0.338377,...,-0.102493,0.306807,0.276972,0.258900,0.236184,0.237291,0.229073,0.211530,0.216900,0.256331
percent_change,0.893572,1.000000,-0.018403,0.008665,0.009581,-0.017961,0.032051,0.158959,-0.094795,0.273260,...,-0.079182,0.326243,0.290948,0.275975,0.250535,0.255451,0.246836,0.229878,0.232345,0.246049
matching_volume,-0.016419,-0.018403,1.000000,0.627757,0.534907,0.923316,0.860472,-0.046468,0.120132,-0.182444,...,0.285065,0.005722,0.016842,0.027839,0.046024,0.025251,0.034274,0.050197,0.032386,0.029340
negotiate_volume,0.024262,0.008665,0.627757,1.000000,0.811036,0.586024,0.548093,-0.045510,0.099382,-0.149067,...,0.206641,-0.010247,-0.001605,-0.000128,0.014869,-0.005799,0.005808,0.013342,0.002268,-0.005580
negotiate_value,0.025782,0.009581,0.534907,0.811036,1.000000,0.530821,0.469530,-0.070647,0.060042,-0.136920,...,0.194948,-0.003718,0.009718,0.009360,0.020284,0.006191,0.013902,0.018570,0.012477,-0.015150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
close_dema_7_13_direction,0.237291,0.255451,0.025251,-0.005799,0.006191,0.001471,0.041508,0.094633,-0.051686,0.048241,...,-0.056276,0.691771,0.843428,0.938826,0.840540,1.000000,0.933287,0.802011,0.876788,0.637992
close_dema_7_15_direction,0.229073,0.246836,0.034274,0.005808,0.013902,0.008096,0.047748,0.096044,-0.042950,0.042828,...,-0.066214,0.626690,0.778296,0.873752,0.904509,0.933287,1.000000,0.868509,0.942600,0.662923
close_dema_7_19_direction,0.211530,0.229878,0.050197,0.013342,0.018570,0.011148,0.052797,0.104753,-0.025121,0.028768,...,-0.087440,0.514023,0.657869,0.748322,0.948454,0.802011,0.868509,1.000000,0.911389,0.696322
close_dema_8_15_direction,0.216900,0.232345,0.032386,0.002268,0.012477,0.003530,0.043860,0.093977,-0.040757,0.040412,...,-0.066722,0.570207,0.720925,0.816385,0.930225,0.876788,0.942600,0.911389,1.000000,0.678543
